# 03 — MT baseline: Opus-MT VI→EN

Tests the MT stage in isolation (text-to-text, no audio involved), per
CLAUDE.md's model stack baseline: `Helsinki-NLP/opus-mt-vi-en`
(Apache-licensed, per-pair Opus-MT — not NLLB/MADLAD, see the license
trap note in CLAUDE.md).

This notebook:
1. Loads the model in PyTorch and translates a few VI sentences (one
   general, three with industrial-glossary flavor) as a reference.
2. Exports the model to ONNX (encoder + merged decoder-with-past) via
   `optimum`, greedy decode only (beam=1), matching the latency-driven
   decode strategy in CLAUDE.md.
3. Verifies the ONNX Runtime output matches the PyTorch reference
   exactly on the same sentences.
4. Converts Marian's slow tokenizer (sentencepiece + vocab.json — no
   fast/tokenizers.json variant ships for Marian) into a single
   `tokenizer.json` loadable by the Rust `tokenizers` crate, and
   verifies it reproduces the slow tokenizer's token ids exactly.
5. Packages the artifacts into `models/mt/vi-en/` for `src/stages/mt.rs`
   to consume.


In [ ]:
from pathlib import Path

MODEL_ID = "Helsinki-NLP/opus-mt-vi-en"
EXPORT_DIR = Path("../models/mt/vi-en/_export")
FINAL_DIR = Path("../models/mt/vi-en")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 1. PyTorch baseline

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model = MarianMTModel.from_pretrained(MODEL_ID)
model.eval()

# One general sentence + three with industrial/safety vocabulary, since
# that's the domain this device targets (see CLAUDE.md -> Differentiators
# -> Industrial glossary engine).
test_sentences = [
    "Xin chào, bạn khỏe không?",
    "Vui lòng đeo kính bảo hộ trước khi vận hành máy.",
    "Xe nâng đang đến gần khu vực số ba.",
    "Dừng máy ngay lập tức nếu nghe thấy báo động.",
]

pytorch_outputs = []
for sent in test_sentences:
    batch = tokenizer([sent], return_tensors="pt", padding=True)
    gen = model.generate(**batch, max_new_tokens=64, num_beams=1)
    out = tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
    pytorch_outputs.append(out)
    print(f"VI: {sent}\nEN: {out}\n")


**Finding relevant to the glossary engine differentiator:** the generic
model mistranslates domain terms — "xe nâng" (forklift) comes out as
"lift", and "vận hành máy" (operate the machine) comes out as "the
engine works". This is exactly the gap the industrial glossary engine +
LoRA fine-tune is meant to close (CLAUDE.md -> Differentiators #1,
"Glossary term accuracy ≥ 95% enforced" target). Baseline numbers here
are the pre-glossary reference point, not the target quality bar.


## 2. Export to ONNX

In [ ]:
from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path=MODEL_ID,
    output=str(EXPORT_DIR),
    task="text2text-generation-with-past",
)
sorted(p.name for p in EXPORT_DIR.iterdir())


`optimum` exports `encoder_model.onnx` plus three decoder variants
(`decoder_model.onnx`, `decoder_with_past_model.onnx`,
`decoder_model_merged.onnx`). The merged decoder covers both the
first-step (no cache) and subsequent-step (with KV cache) cases behind a
single `use_cache_branch` input, so it's the only decoder file we need
to keep — encoder + merged decoder is the full runtime dependency.


## 3. Verify ONNX Runtime output matches PyTorch

In [ ]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM

ort_model = ORTModelForSeq2SeqLM.from_pretrained(str(EXPORT_DIR), use_cache=True)

all_match = True
for sent, expected in zip(test_sentences, pytorch_outputs):
    batch = tokenizer([sent], return_tensors="pt", padding=True)
    gen = ort_model.generate(**batch, max_new_tokens=64, num_beams=1)
    out = tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
    match = out == expected
    all_match &= match
    print(f"VI: {sent}\nONNX EN: {out}\nmatches PyTorch: {match}\n")

assert all_match, "ONNX output diverged from the PyTorch reference"
print("ONNX output matches PyTorch exactly on all test sentences.")


## 4. Convert the tokenizer for Rust

Marian ships a *slow* tokenizer only (sentencepiece `source.spm` /
`target.spm` + a shared `vocab.json`, `separate_vocabs=false`) — there's
no fast `tokenizer.json` variant, and `transformers`' generic slow→fast
converter doesn't have a registered path for Marian. The Rust
`tokenizers` crate is pure Rust (no C++ sentencepiece dependency, unlike
the `sentencepiece` crate which needs `cmake` — confirmed unavailable in
this environment), so we build the fast tokenizer once, here, in Python:

- **Vocab / ids**: `vocab.json` is the authority for token ids (not
  `source.spm`'s own internal ordering — they differ, since `vocab.json`
  reassigns `</s>`/`<unk>` to ids 0/1 and drops `<s>` entirely). We build
  the Unigram vocab list ordered by `vocab.json`'s ids, pulling
  unigram-LM scores from `source.spm` by piece string.
- **Normalization**: `source.spm`'s `normalizer_spec.precompiled_charsmap`
  is extracted directly from the raw protobuf and used as a `Precompiled`
  normalizer — the exact same normalization table sentencepiece itself
  uses, not an approximation.
- **Segmentation**: standard Metaspace pre-tokenizer/decoder
  (`▁` = word-boundary marker), which is what sentencepiece's Unigram
  mode uses.

Because `separate_vocabs=false`, this single tokenizer covers both
directions: VI text → ids (encoder input) and ids → EN text (decoder
output decode) share the same vocab. `target.spm`'s scores are unused —
decoding is just id→piece lookup + Metaspace join, no segmentation scoring
involved.


In [ ]:
import json

import sentencepiece as spm
import sentencepiece.sentencepiece_model_pb2 as spm_pb2
from tokenizers import Tokenizer, decoders, pre_tokenizers
from tokenizers.models import Unigram
from tokenizers.normalizers import Precompiled, Replace, Sequence

with open(EXPORT_DIR / "vocab.json") as f:
    vocab_json = json.load(f)

vocab_size = max(vocab_json.values()) + 1
id_to_piece = [None] * vocab_size
for piece, idx in vocab_json.items():
    id_to_piece[idx] = piece
assert all(p is not None for p in id_to_piece), "gap in vocab id space"

sp = spm.SentencePieceProcessor(model_file=str(EXPORT_DIR / "source.spm"))
piece_scores = {sp.id_to_piece(i): sp.get_score(i) for i in range(sp.get_piece_size())}
min_score = min(piece_scores.values())

vocab_list = [(piece, piece_scores.get(piece, min_score - 1.0)) for piece in id_to_piece]
unk_id = vocab_json["<unk>"]

fast_tokenizer = Tokenizer(Unigram(vocab_list, unk_id=unk_id, byte_fallback=False))

proto = spm_pb2.ModelProto()
proto.ParseFromString((EXPORT_DIR / "source.spm").read_bytes())
charsmap = proto.normalizer_spec.precompiled_charsmap
fast_tokenizer.normalizer = Sequence([Precompiled(charsmap), Replace(" {2,}", " ")])
fast_tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [pre_tokenizers.Metaspace(replacement="\u2581", prepend_scheme="always")]
)
fast_tokenizer.decoder = decoders.Metaspace(replacement="\u2581", prepend_scheme="always")

(EXPORT_DIR / "tokenizer.json").write_text("")  # placeholder so save() below can overwrite cleanly
fast_tokenizer.save(str(EXPORT_DIR / "tokenizer.json"))
print(f"vocab_size={vocab_size} unk_id={unk_id}")


In [ ]:
all_ok = True

for sent in test_sentences:
    slow_ids = tokenizer(sent).input_ids
    slow_ids = slow_ids[:-1] if slow_ids[-1] == tokenizer.eos_token_id else slow_ids
    fast_ids = fast_tokenizer.encode(sent).ids
    ok = slow_ids == fast_ids
    all_ok &= ok
    print(f"VI encode match: {ok}  ({sent})")

for text in pytorch_outputs:
    ids = tokenizer(text).input_ids[:-1]
    fast_decoded = fast_tokenizer.decode(ids).strip()
    slow_decoded = tokenizer.decode(ids, skip_special_tokens=True).strip()
    ok = fast_decoded == slow_decoded == text
    all_ok &= ok
    print(f"EN decode match: {ok}  ({fast_decoded!r})")

assert all_ok, "converted tokenizer diverged from the slow MarianTokenizer"
print("Converted tokenizer matches the slow MarianTokenizer exactly.")


## 5. Package artifacts for `src/`

In [ ]:
import shutil
from datetime import date

FINAL_DIR.mkdir(parents=True, exist_ok=True)

keep = ["encoder_model.onnx", "decoder_model_merged.onnx", "tokenizer.json"]
for name in keep:
    shutil.copyfile(EXPORT_DIR / name, FINAL_DIR / name)

manifest = f"""# mt/vi-en

- Source checkpoint: {MODEL_ID}
- License: Apache 2.0 (Opus-MT / Helsinki-NLP — safe to commercialize,
  unlike NLLB's CC-BY-NC; see CLAUDE.md license trap note)
- Exported: {date.today().isoformat()}
- Precision: fp32 (baseline export, not yet quantized — INT8 via AI Hub
  Workbench is the documented upgrade path in CLAUDE.md's model stack
  table; revisit before flashing to device, current size is ~500MB,
  well over the ~300MB/pair memory budget target)
- Decode: greedy (beam=1), matches CLAUDE.md's latency-driven decode
  strategy
- Eval at export time: exact-match against PyTorch reference on 4 VI
  test sentences (1 general + 3 industrial-glossary-flavored) — see
  experiments/03_mt.ipynb section 3. No BLEU/domain benchmark run yet
  (tracked in docs/TODO.md).
- Known gap: generic model mistranslates domain terms (e.g. "xe nâng" /
  forklift -> "lift") — this is what the glossary engine + LoRA
  fine-tune are meant to fix, not yet applied to this artifact.

Files:
- `encoder_model.onnx`
- `decoder_model_merged.onnx` (covers both the no-cache first step and
  the with-cache subsequent steps via `use_cache_branch`)
- `tokenizer.json` (Rust `tokenizers`-crate-compatible; see notebook
  section 4 for how it was derived from the slow MarianTokenizer)
"""
(FINAL_DIR / "MANIFEST.md").write_text(manifest)

shutil.rmtree(EXPORT_DIR)
sorted(p.name for p in FINAL_DIR.iterdir())


## 6. Manual greedy KV-cache decode (reference for the Rust port)

`optimum`'s `ORTModelForSeq2SeqLM.generate()` (section 3) hides the actual
ONNX input/output wiring — the Rust port in `src/stages/mt.rs` has to
reimplement that wiring by hand with the `ort` crate. Before writing any
Rust, this section reimplements the same greedy KV-cache decode loop
directly against `onnxruntime`, using the packaged artifacts in
`models/mt/vi-en/` (the fast tokenizer + the two ONNX files), so the
exact algorithm is nailed down and verified against the PyTorch
reference first.

Getting there surfaced two real gotchas, both load-bearing for the Rust
implementation:

1. **Execution provider matters.** This machine exposes
   `CoreMLExecutionProvider` (macOS), and `onnxruntime` picks it over CPU
   by default. It mishandles this graph's zero-length (`seq_len=0`) dummy
   KV-cache tensors on the first decode step — hard failure inside the
   exported `If` node's `MatMul`. Forcing `providers=["CPUExecutionProvider"]`
   fixes it. (Irrelevant on the actual QCS6490 target, which routes
   through the QNN execution provider — but worth knowing here, and the
   general lesson — pin the execution provider explicitly rather than
   trusting the default — carries over.)
2. **The merged decoder's cross-attention present outputs are only valid
   on the first step.** `decoder_model_merged.onnx` has a `use_cache_branch`
   input that switches between "compute cross-attention K/V fresh from
   `encoder_hidden_states`" (step 0) and "reuse cached K/V" (later steps).
   The `present.*.encoder.key/value` outputs are only correctly populated
   on step 0; blindly re-reading them every step (as the naive symmetric
   implementation does) picks up malformed tensors on step 2 onward and
   the graph fails. Fix: capture `present.*.encoder.*` once after step 0
   and hold those tensors fixed for the rest of decoding — never refresh
   them from later outputs. This matches what `optimum` itself does
   internally (see `ORTDecoderForSeq2Seq.forward` in
   `optimum/onnxruntime/modeling_seq2seq.py`), it's just not obvious from
   the ONNX graph's I/O signature alone.


In [ ]:
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer as FastTokenizer

NUM_LAYERS, NUM_HEADS, HEAD_DIM = 6, 8, 64
DECODER_START_TOKEN_ID = 53738  # == pad_token_id, see generation_config
EOS_TOKEN_ID = 0
PAD_TOKEN_ID = 53738  # bad_words_ids in generation_config -- never generate this

rt_tokenizer = FastTokenizer.from_file(str(FINAL_DIR / "tokenizer.json"))
rt_enc_sess = ort.InferenceSession(
    str(FINAL_DIR / "encoder_model.onnx"), providers=["CPUExecutionProvider"]
)
rt_dec_sess = ort.InferenceSession(
    str(FINAL_DIR / "decoder_model_merged.onnx"), providers=["CPUExecutionProvider"]
)
rt_dec_output_names = [o.name for o in rt_dec_sess.get_outputs()]


def greedy_translate(text: str, max_new_tokens: int = 64) -> str:
    ids = rt_tokenizer.encode(text).ids + [EOS_TOKEN_ID]
    input_ids = np.array([ids], dtype=np.int64)
    attention_mask = np.ones_like(input_ids)
    encoder_hidden_states = rt_enc_sess.run(
        None, {"input_ids": input_ids, "attention_mask": attention_mask}
    )[0]

    empty_kv = np.zeros((1, NUM_HEADS, 0, HEAD_DIM), dtype=np.float32)
    past_dec_k = [empty_kv] * NUM_LAYERS
    past_dec_v = [empty_kv] * NUM_LAYERS
    past_enc_k = [empty_kv] * NUM_LAYERS
    past_enc_v = [empty_kv] * NUM_LAYERS

    next_id = DECODER_START_TOKEN_ID
    generated = []

    for step in range(max_new_tokens):
        feeds = {
            "input_ids": np.array([[next_id]], dtype=np.int64),
            "encoder_attention_mask": attention_mask,
            "encoder_hidden_states": encoder_hidden_states,
            "use_cache_branch": np.array([step > 0], dtype=bool),
        }
        for i in range(NUM_LAYERS):
            feeds[f"past_key_values.{i}.decoder.key"] = past_dec_k[i]
            feeds[f"past_key_values.{i}.decoder.value"] = past_dec_v[i]
            feeds[f"past_key_values.{i}.encoder.key"] = past_enc_k[i]
            feeds[f"past_key_values.{i}.encoder.value"] = past_enc_v[i]

        out = dict(zip(rt_dec_output_names, rt_dec_sess.run(rt_dec_output_names, feeds)))

        logits = out["logits"][0, -1, :].copy()
        logits[PAD_TOKEN_ID] = -np.inf
        next_id = int(np.argmax(logits))
        if next_id == EOS_TOKEN_ID:
            break
        generated.append(next_id)

        for i in range(NUM_LAYERS):
            past_dec_k[i] = out[f"present.{i}.decoder.key"]
            past_dec_v[i] = out[f"present.{i}.decoder.value"]
        if step == 0:
            # See gotcha #2 above: only valid on the first step.
            for i in range(NUM_LAYERS):
                past_enc_k[i] = out[f"present.{i}.encoder.key"]
                past_enc_v[i] = out[f"present.{i}.encoder.value"]

    return rt_tokenizer.decode(generated)


all_match = True
for sent, expected in zip(test_sentences, pytorch_outputs):
    out = greedy_translate(sent)
    match = out == expected
    all_match &= match
    print(f"VI: {sent}\nmanual ort greedy: {out}\nmatches PyTorch: {match}\n")

assert all_match, "manual greedy decode diverged from the PyTorch reference"
print("Manual greedy KV-cache decode matches PyTorch exactly -- this is the algorithm to port to Rust.")


## 7. Glossary enforcement: decoder sequence-bias (prototype)

`docs/decisions/glossary-runtime-enforcement-2026-07.md` already ruled
out placeholder substitution (unreliable — 4/6 candidate placeholders
got silently mangled by the tokenizer). This section tries a different,
cheaper-than-fine-tuning technique that stays compatible with greedy
(beam=1) decoding: **bias the decoder logits toward the glossary
target term's tokens whenever the model is in a position to start (or
continue) emitting them**, rather than forcing them at a specific
position. This is a simplified version of what HF `generate()` calls
`sequence_bias` — implemented by hand here, directly against the same
manual ONNX Runtime greedy-decode loop from section 6, so the technique
is validated against the *exact* algorithm `src/stages/mt.rs` runs, not
against a higher-level HF convenience API that might hide details that
don't port to Rust.

Mechanism: at every decode step, for each glossary target term relevant
to this source sentence (detected by a case-insensitive substring match
of the *source* term against the input text), tokenize the target term
into an id sequence `seq`. If the tokens generated so far end with
`seq[:-1]` (trivially true when `len(seq) == 1`, i.e. "any position"),
add a fixed bias to `seq[-1]`'s logit before the argmax. No beam search,
no change to latency-critical control flow — one extra pass over a
handful of short id sequences per decode step.


In [ ]:
import csv

# vi -> en glossary dict, loaded from the real termbase (same file
# src/stages/glossary.rs reads).
glossary_vi_en = {}
with open("../data/glossary/termbase.tsv", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        glossary_vi_en[row["vi"]] = row["en"]

glossary_vi_en


def find_glossary_bias_seqs(text: str, glossary: dict) -> list[tuple[str, str, list[int]]]:
    text_lower = text.lower()
    matches = []
    for src_term, tgt_term in glossary.items():
        if src_term.lower() in text_lower:
            seq = rt_tokenizer.encode(tgt_term).ids
            matches.append((src_term, tgt_term, seq))
    return matches


def biased_logits(logits, generated, bias_seqs, bias_value):
    """Bias every token position `k` of every sequence in `bias_seqs`,
    not just the last one -- a sequence must be biased into *starting*
    (k=0, prefix is empty, always eligible) as well as continuing
    (k>0, prefix = seq[:k] must match the tail of `generated`). An
    earlier version of this only handled continuation, so multi-token
    terms (e.g. "forklift" = [_fork, lift]) never got their first token
    biased and the sequence could never start -- caught by checking the
    raw per-step logits below rather than trusting the first result.
    """
    for seq in bias_seqs:
        for k in range(len(seq)):
            prefix = seq[:k]
            if k == 0 or generated[-k:] == prefix:
                logits[seq[k]] += bias_value
    return logits


def greedy_translate_biased(
    text: str, glossary: dict, bias_value: float = 6.0, max_new_tokens: int = 64, verbose: bool = False
) -> str:
    matches = find_glossary_bias_seqs(text, glossary)
    bias_seqs = [seq for _, _, seq in matches]
    if verbose and matches:
        print("  glossary matches:", [(s, t) for s, t, _ in matches])

    ids = rt_tokenizer.encode(text).ids + [EOS_TOKEN_ID]
    input_ids = np.array([ids], dtype=np.int64)
    attention_mask = np.ones_like(input_ids)
    encoder_hidden_states = rt_enc_sess.run(
        None, {"input_ids": input_ids, "attention_mask": attention_mask}
    )[0]

    empty_kv = np.zeros((1, NUM_HEADS, 0, HEAD_DIM), dtype=np.float32)
    past_dec_k = [empty_kv] * NUM_LAYERS
    past_dec_v = [empty_kv] * NUM_LAYERS
    past_enc_k = [empty_kv] * NUM_LAYERS
    past_enc_v = [empty_kv] * NUM_LAYERS

    next_id = DECODER_START_TOKEN_ID
    generated = []

    for step in range(max_new_tokens):
        feeds = {
            "input_ids": np.array([[next_id]], dtype=np.int64),
            "encoder_attention_mask": attention_mask,
            "encoder_hidden_states": encoder_hidden_states,
            "use_cache_branch": np.array([step > 0], dtype=bool),
        }
        for i in range(NUM_LAYERS):
            feeds[f"past_key_values.{i}.decoder.key"] = past_dec_k[i]
            feeds[f"past_key_values.{i}.decoder.value"] = past_dec_v[i]
            feeds[f"past_key_values.{i}.encoder.key"] = past_enc_k[i]
            feeds[f"past_key_values.{i}.encoder.value"] = past_enc_v[i]

        out = dict(zip(rt_dec_output_names, rt_dec_sess.run(rt_dec_output_names, feeds)))

        logits = out["logits"][0, -1, :].copy()
        logits[PAD_TOKEN_ID] = -np.inf
        logits = biased_logits(logits, generated, bias_seqs, bias_value)

        next_id = int(np.argmax(logits))
        if next_id == EOS_TOKEN_ID:
            break
        generated.append(next_id)

        for i in range(NUM_LAYERS):
            past_dec_k[i] = out[f"present.{i}.decoder.key"]
            past_dec_v[i] = out[f"present.{i}.decoder.value"]
        if step == 0:
            for i in range(NUM_LAYERS):
                past_enc_k[i] = out[f"present.{i}.encoder.key"]
                past_enc_v[i] = out[f"present.{i}.encoder.value"]

    return rt_tokenizer.decode(generated)


# Sanity check: bias_value=0 (or no glossary matches) must reproduce
# section 6's unbiased output exactly -- otherwise the bias plumbing
# itself has a bug, independent of whether biasing "works".
for sent, expected in zip(test_sentences, pytorch_outputs):
    out = greedy_translate_biased(sent, {}, bias_value=6.0)
    assert out == expected, f"unbiased-path regression on {sent!r}: got {out!r}, expected {expected!r}"
print("no-op check passed: empty glossary reproduces the section-6 baseline exactly.\n")

# The known failure case from section 1: "xe nâng" (forklift) mistranslated as "lift".
# Diagnostic (raw ort, not this loop): at the decode step where the
# baseline picks "_lift" (id 5327, logit 5.72), "_fork" (id 16048, the
# first token of "forklift") sits at logit -3.74 -- a gap of 9.46, so
# bias values below ~9.5 are mathematically guaranteed not to flip it.
known_failure = "Xe nâng đang đến gần khu vực số ba."
print(f"baseline (no bias): {greedy_translate(known_failure)}")
for bias_value in [2.0, 4.0, 6.0, 8.0, 9.5, 10.0, 12.0, 14.0]:
    out = greedy_translate_biased(known_failure, glossary_vi_en, bias_value=bias_value)
    print(f"bias={bias_value:>4.1f}: {out}")


In [ ]:
# One real carrier sentence per termbase entry (not just the 1 known
# failure case) -- checks whether biasing generalizes across the whole
# glossary, not just the term it was designed against. Picked a
# bias_value from the sweep above once printed.
carrier_sentences = {
    "forklift": "Xe nâng đang đến gần khu vực số ba.",
    "stop_the_machine": "Dừng máy ngay lập tức nếu nghe thấy báo động.",
    "safety_glasses": "Vui lòng đeo kính bảo hộ trước khi vận hành máy.",
    "alarm": "Dừng máy ngay lập tức nếu nghe thấy báo động.",
    "emergency_exit": "Lối thoát hiểm ở phía bên trái.",
    "fire_extinguisher": "Bình chữa cháy nằm gần cửa ra vào.",
    "helmet": "Vui lòng đội mũ bảo hộ khi vào công trường.",
    "danger": "Khu vực này rất nguy hiểm.",
    "loading_dock": "Xe tải đang đậu ở khu vực bốc xếp.",
    "first_aid": "Anh ấy đang thực hiện sơ cứu.",
}

BIAS_VALUE = 8.0  # set after inspecting the sweep printed above

rows = []
for concept, sent in carrier_sentences.items():
    matches = find_glossary_bias_seqs(sent, glossary_vi_en)
    assert matches, f"carrier sentence for {concept!r} doesn't contain its own source term: {sent!r}"
    target_term = matches[0][1]

    baseline = greedy_translate(sent)
    biased = greedy_translate_biased(sent, glossary_vi_en, bias_value=BIAS_VALUE)

    baseline_hit = target_term.lower() in baseline.lower()
    biased_hit = target_term.lower() in biased.lower()
    rows.append((concept, target_term, baseline_hit, biased_hit, baseline, biased))

    flag = "OK " if biased_hit else "MISS"
    print(f"[{flag}] {concept:<20} target={target_term!r}")
    print(f"       baseline ({'hit' if baseline_hit else 'miss'}): {baseline}")
    print(f"       biased   ({'hit' if biased_hit else 'miss'}): {biased}\n")

n = len(rows)
baseline_acc = sum(r[2] for r in rows) / n
biased_acc = sum(r[3] for r in rows) / n
print(f"baseline term accuracy: {baseline_acc:.0%} ({sum(r[2] for r in rows)}/{n})")
print(f"biased term accuracy:   {biased_acc:.0%} ({sum(r[3] for r in rows)}/{n})")


### 7b. Fixing the repetition-loop failure mode

The sweep above shows the mechanism genuinely works — at `bias>=9.5` the
model does switch to "forklift" — but the bias is applied unconditionally
at every remaining step, including *after* the term has already been
correctly emitted once. Since `k=0` (the sequence's first token) is
always eligible, the model keeps getting shoved back toward "fork" long
after "forklift" is done, producing "fork fork fork fork..." until
`max_new_tokens` cuts it off. Fix: track which glossary sequences have
already appeared once in `generated`, and stop biasing those — a
sequence gets one shot to insert itself, not a standing preference for
the rest of decoding.


In [ ]:
def greedy_translate_biased_v2(
    text: str, glossary: dict, bias_value: float = 10.0, max_new_tokens: int = 64
) -> str:
    matches = find_glossary_bias_seqs(text, glossary)
    bias_seqs = [seq for _, _, seq in matches]
    satisfied = [False] * len(bias_seqs)

    ids = rt_tokenizer.encode(text).ids + [EOS_TOKEN_ID]
    input_ids = np.array([ids], dtype=np.int64)
    attention_mask = np.ones_like(input_ids)
    encoder_hidden_states = rt_enc_sess.run(
        None, {"input_ids": input_ids, "attention_mask": attention_mask}
    )[0]

    empty_kv = np.zeros((1, NUM_HEADS, 0, HEAD_DIM), dtype=np.float32)
    past_dec_k = [empty_kv] * NUM_LAYERS
    past_dec_v = [empty_kv] * NUM_LAYERS
    past_enc_k = [empty_kv] * NUM_LAYERS
    past_enc_v = [empty_kv] * NUM_LAYERS

    next_id = DECODER_START_TOKEN_ID
    generated = []

    for step in range(max_new_tokens):
        feeds = {
            "input_ids": np.array([[next_id]], dtype=np.int64),
            "encoder_attention_mask": attention_mask,
            "encoder_hidden_states": encoder_hidden_states,
            "use_cache_branch": np.array([step > 0], dtype=bool),
        }
        for i in range(NUM_LAYERS):
            feeds[f"past_key_values.{i}.decoder.key"] = past_dec_k[i]
            feeds[f"past_key_values.{i}.decoder.value"] = past_dec_v[i]
            feeds[f"past_key_values.{i}.encoder.key"] = past_enc_k[i]
            feeds[f"past_key_values.{i}.encoder.value"] = past_enc_v[i]

        out = dict(zip(rt_dec_output_names, rt_dec_sess.run(rt_dec_output_names, feeds)))

        logits = out["logits"][0, -1, :].copy()
        logits[PAD_TOKEN_ID] = -np.inf
        for seq, done in zip(bias_seqs, satisfied):
            if done:
                continue
            for k in range(len(seq)):
                prefix = seq[:k]
                if k == 0 or generated[-k:] == prefix:
                    logits[seq[k]] += bias_value

        next_id = int(np.argmax(logits))
        if next_id == EOS_TOKEN_ID:
            break
        generated.append(next_id)

        for i, seq in enumerate(bias_seqs):
            if not satisfied[i] and len(generated) >= len(seq) and generated[-len(seq):] == seq:
                satisfied[i] = True

        for i in range(NUM_LAYERS):
            past_dec_k[i] = out[f"present.{i}.decoder.key"]
            past_dec_v[i] = out[f"present.{i}.decoder.value"]
        if step == 0:
            for i in range(NUM_LAYERS):
                past_enc_k[i] = out[f"present.{i}.encoder.key"]
                past_enc_v[i] = out[f"present.{i}.encoder.value"]

    return rt_tokenizer.decode(generated)


print(f"baseline (no bias): {greedy_translate(known_failure)}")
for bias_value in [9.5, 10.0, 12.0, 14.0, 18.0]:
    out = greedy_translate_biased_v2(known_failure, glossary_vi_en, bias_value=bias_value)
    print(f"bias={bias_value:>4.1f}: {out}")


In [ ]:
BIAS_VALUE_V2 = 12.0  # picked from the sweep above

rows_v2 = []
for concept, sent in carrier_sentences.items():
    matches = find_glossary_bias_seqs(sent, glossary_vi_en)
    target_term = matches[0][1]

    baseline = greedy_translate(sent)
    biased = greedy_translate_biased_v2(sent, glossary_vi_en, bias_value=BIAS_VALUE_V2)

    baseline_hit = target_term.lower() in baseline.lower()
    biased_hit = target_term.lower() in biased.lower()
    rows_v2.append((concept, target_term, baseline_hit, biased_hit, baseline, biased))

    flag = "OK " if biased_hit else "MISS"
    print(f"[{flag}] {concept:<20} target={target_term!r}")
    print(f"       baseline ({'hit' if baseline_hit else 'miss'}): {baseline}")
    print(f"       biased   ({'hit' if biased_hit else 'miss'}): {biased}\n")

n = len(rows_v2)
baseline_acc = sum(r[2] for r in rows_v2) / n
biased_acc = sum(r[3] for r in rows_v2) / n
print(f"baseline term accuracy: {baseline_acc:.0%} ({sum(r[2] for r in rows_v2)}/{n})")
print(f"biased (v2) term accuracy: {biased_acc:.0%} ({sum(r[3] for r in rows_v2)}/{n})")


### 7c. `loading_dock` and `first_aid` reveal one more bug

Both looped ("loading loading loading...", "first first first...") in
7b's results. Cause: 7b still checks *every* `k` independently and
biases all matching ones simultaneously — once a sequence is one token
in (`progress=1`), both `k=0` ("start the sequence again") and `k=1`
("continue it") get biased on the same step, and they compete. If the
model's natural preference for repeating the first word is strong
enough, `k=0` keeps winning over `k=1` for many steps in a row. The
correct version tracks a single current match position per sequence
(standard prefix-matching, not independent per-k checks) — at any time
there is exactly one next expected token to bias, not several.


In [ ]:
def greedy_translate_biased_v3(
    text: str, glossary: dict, bias_value: float = 10.0, max_new_tokens: int = 64
) -> str:
    matches = find_glossary_bias_seqs(text, glossary)
    bias_seqs = [seq for _, _, seq in matches]
    satisfied = [False] * len(bias_seqs)
    progress = [0] * len(bias_seqs)  # index of the next expected token in each seq

    ids = rt_tokenizer.encode(text).ids + [EOS_TOKEN_ID]
    input_ids = np.array([ids], dtype=np.int64)
    attention_mask = np.ones_like(input_ids)
    encoder_hidden_states = rt_enc_sess.run(
        None, {"input_ids": input_ids, "attention_mask": attention_mask}
    )[0]

    empty_kv = np.zeros((1, NUM_HEADS, 0, HEAD_DIM), dtype=np.float32)
    past_dec_k = [empty_kv] * NUM_LAYERS
    past_dec_v = [empty_kv] * NUM_LAYERS
    past_enc_k = [empty_kv] * NUM_LAYERS
    past_enc_v = [empty_kv] * NUM_LAYERS

    next_id = DECODER_START_TOKEN_ID
    generated = []

    for step in range(max_new_tokens):
        feeds = {
            "input_ids": np.array([[next_id]], dtype=np.int64),
            "encoder_attention_mask": attention_mask,
            "encoder_hidden_states": encoder_hidden_states,
            "use_cache_branch": np.array([step > 0], dtype=bool),
        }
        for i in range(NUM_LAYERS):
            feeds[f"past_key_values.{i}.decoder.key"] = past_dec_k[i]
            feeds[f"past_key_values.{i}.decoder.value"] = past_dec_v[i]
            feeds[f"past_key_values.{i}.encoder.key"] = past_enc_k[i]
            feeds[f"past_key_values.{i}.encoder.value"] = past_enc_v[i]

        out = dict(zip(rt_dec_output_names, rt_dec_sess.run(rt_dec_output_names, feeds)))

        logits = out["logits"][0, -1, :].copy()
        logits[PAD_TOKEN_ID] = -np.inf
        for i, seq in enumerate(bias_seqs):
            if not satisfied[i]:
                logits[seq[progress[i]]] += bias_value  # exactly one biased token per sequence

        next_id = int(np.argmax(logits))
        if next_id == EOS_TOKEN_ID:
            break
        generated.append(next_id)

        for i, seq in enumerate(bias_seqs):
            if satisfied[i]:
                continue
            if next_id == seq[progress[i]]:
                progress[i] += 1
                if progress[i] == len(seq):
                    satisfied[i] = True
            else:
                # Didn't continue -- restart the match if this token
                # happens to also be the sequence's first token,
                # otherwise back to square one.
                progress[i] = 1 if (seq[0] == next_id and len(seq) > 1) else 0

        for i in range(NUM_LAYERS):
            past_dec_k[i] = out[f"present.{i}.decoder.key"]
            past_dec_v[i] = out[f"present.{i}.decoder.value"]
        if step == 0:
            for i in range(NUM_LAYERS):
                past_enc_k[i] = out[f"present.{i}.encoder.key"]
                past_enc_v[i] = out[f"present.{i}.encoder.value"]

    return rt_tokenizer.decode(generated)


print(f"baseline (no bias): {greedy_translate(known_failure)}")
for bias_value in [9.5, 10.0, 12.0, 14.0]:
    out = greedy_translate_biased_v3(known_failure, glossary_vi_en, bias_value=bias_value)
    print(f"bias={bias_value:>4.1f}: {out}")

print()
BIAS_VALUE_V3 = 12.0
rows_v3 = []
for concept, sent in carrier_sentences.items():
    matches = find_glossary_bias_seqs(sent, glossary_vi_en)
    target_term = matches[0][1]

    baseline = greedy_translate(sent)
    biased = greedy_translate_biased_v3(sent, glossary_vi_en, bias_value=BIAS_VALUE_V3)

    baseline_hit = target_term.lower() in baseline.lower()
    biased_hit = target_term.lower() in biased.lower()
    rows_v3.append((concept, target_term, baseline_hit, biased_hit, baseline, biased))

    flag = "OK " if biased_hit else "MISS"
    print(f"[{flag}] {concept:<20} target={target_term!r}")
    print(f"       baseline ({'hit' if baseline_hit else 'miss'}): {baseline}")
    print(f"       biased   ({'hit' if biased_hit else 'miss'}): {biased}\n")

n = len(rows_v3)
baseline_acc = sum(r[2] for r in rows_v3) / n
biased_acc = sum(r[3] for r in rows_v3) / n
print(f"baseline term accuracy: {baseline_acc:.0%} ({sum(r[2] for r in rows_v3)}/{n})")
print(f"biased (v3) term accuracy: {biased_acc:.0%} ({sum(r[3] for r in rows_v3)}/{n})")


### 7d. v3 fixes term accuracy but costs fluency — only bias when needed

v3 hits 10/10 on the term-presence metric, but several previously-fine
translations got visibly worse ("This area is dangerous." → "danger
zone.", "He's doing his first aid." → "He first-aid first aid.") —
biasing ran even on the 5 sentences the baseline already translated
correctly, and cost them grammar for no benefit. Cheap fix: run the
*unbiased* pass first (it's what `mt.rs` already does, ~60ms per
CLAUDE.md's real pipeline run), check whether every matched glossary
target term is already present in that output, and only fall back to
the biased decode — a second full decode pass — when something's
actually missing. Doubles latency only on the sentences that need
correcting, not on every glossary-flavored sentence.


In [ ]:
def glossary_enforced_translate(text: str, glossary: dict, bias_value: float = 12.0) -> tuple[str, bool]:
    """Returns (translation, used_biased_pass)."""
    matches = find_glossary_bias_seqs(text, glossary)
    baseline = greedy_translate(text)
    if not matches:
        return baseline, False
    if all(tgt_term.lower() in baseline.lower() for _, tgt_term, _ in matches):
        return baseline, False
    return greedy_translate_biased_v3(text, glossary, bias_value=bias_value), True


rows_v4 = []
for concept, sent in carrier_sentences.items():
    matches = find_glossary_bias_seqs(sent, glossary_vi_en)
    target_term = matches[0][1]

    baseline = greedy_translate(sent)
    result, used_bias = glossary_enforced_translate(sent, glossary_vi_en)

    baseline_hit = target_term.lower() in baseline.lower()
    result_hit = target_term.lower() in result.lower()
    rows_v4.append((concept, target_term, baseline_hit, result_hit, baseline, result, used_bias))

    flag = "OK " if result_hit else "MISS"
    pass_used = "biased pass" if used_bias else "unbiased (unchanged)"
    print(f"[{flag}] {concept:<20} target={target_term!r}  ({pass_used})")
    print(f"       baseline: {baseline}")
    print(f"       result:   {result}\n")

n = len(rows_v4)
result_acc = sum(r[3] for r in rows_v4) / n
biased_pass_count = sum(r[6] for r in rows_v4)
print(f"baseline term accuracy: {sum(r[2] for r in rows_v4) / n:.0%} ({sum(r[2] for r in rows_v4)}/{n})")
print(f"final term accuracy:    {result_acc:.0%} ({sum(r[3] for r in rows_v4)}/{n})")
print(f"biased pass triggered on {biased_pass_count}/{n} sentences "
      f"(the other {n - biased_pass_count} kept their unbiased translation untouched)")
